In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import re
import sys
import shutil
import zipfile
import requests
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm
from urllib.parse import urlparse, parse_qs
from tqdm.notebook import tqdm
sys.path.append(r'E:\repository\dataset_tools\isds_tool\PS_data')
from vis import select_defect, esresult2yolo, esimage_merge
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


In [3]:
root_dir = r"\\158.132.186.40\isds\huilin\isds\environ_sense_data\task1113"
RESULT_ALL = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportResults?subProjectId="
RESULT_JSON = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportOutputResults?subProjectId="
CONFIG_EXPORT = "http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/getSubprojectMetaData?subProjectId="

data_dir = os.path.join(root_dir, 'task1113', 'results')
subproject_list = [
    485,
]

merge_dir = os.path.join(root_dir, 'merge_dir')
gap_num=3

In [5]:
# def extract_date_from_project_name(project_name):
#     pattern = r"^2025(\d{4})_SIT$"
#     match = re.match(pattern, project_name)
    
#     if match:
#         date = match.group(1)  # 提取 xxxx
#         return date
#     else:
#         return 0000

def extract_date_from_project_name(project_name):
    pattern = r"^Project: 2025(\d{4})$"
    match = re.match(pattern, project_name)
    
    if match:
        date = match.group(1)  # 提取 xxxx
        return date
    else:
        return 0000

In [6]:
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

# 多线程下载函数
def download_zip_files_mp(subproject_list, root_dir, overwirte=False, max_workers=4):
    zip_download_dict = {}

    def download_single(subproject_id):
        # 每个线程创建独立的session
        session = requests.Session()
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        })

        download_url = RESULT_ALL+str(subproject_id)
        config_url = CONFIG_EXPORT+str(subproject_id)

        try:
            print(f"🔗 正在连接: {config_url}")
            response = session.get(config_url)
            response.raise_for_status()
            data = response.json()
            project_name = data['project']['name']
            date = extract_date_from_project_name(project_name)

            save_dir = os.path.join(root_dir, 'task'+date, 'results')
            os.makedirs(save_dir, exist_ok=True)

            print(f"🔗 正在连接: {download_url}")
            response = session.get(download_url, stream=True)
            response.raise_for_status()

            total_size = int(response.headers.get("content-length", 0))
            chunk_size = 1024 * 1024  # 1MB

            print(f'total size: {total_size/chunk_size} MB')

            parsed_url = urlparse(download_url)
            query_params = parse_qs(parsed_url.query)
            sub_project_id = query_params.get('subProjectId', ['unknown'])[0]
            filename = f"{sub_project_id}.zip"
            save_path = os.path.join(save_dir, filename)

            if not os.path.exists(save_path) or overwirte:
                with open(save_path, 'wb') as f, tqdm(
                    total=total_size,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=f"📥 下载 {filename} -> {save_dir}",
                    leave=True,
                ) as pbar:
                    for chunk in response.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

                print(f"✅ 下载完成: {save_path}\n")
                return subproject_id, save_path
            else:
                print(f"⚠️ 文件已存在，跳过下载: {save_path}\n")
                return subproject_id, save_path
        except Exception as e:
            print(f"❌ 下载失败: {download_url}\n原因: {e}")
            return subproject_id, None

    # 使用线程池执行下载任务
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 提交所有任务
        futures = {executor.submit(download_single, sid): sid for sid in subproject_list}

        # 收集结果
        for future in concurrent.futures.as_completed(futures):
            subproject_id, save_path = future.result()
            if save_path:
                zip_download_dict[subproject_id] = save_path

    return zip_download_dict

In [7]:
# zip_download_dict = download_zip_files(subproject_list, root_dir)
zip_download_dict = download_zip_files_mp(subproject_list, root_dir)

🔗 正在连接: http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/getSubprojectMetaData?subProjectId=485
❌ 下载失败: http://ec2-54-46-0-164.ap-east-1.compute.amazonaws.com/emes/api/v2/workflow-result/exportResults?subProjectId=485
原因: can only concatenate str (not "int") to str


In [ ]:
def simple_unzip(zip_path, dst_dir):
    print(f'{zip_path} unzip...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dst_dir)
    print(f'{zip_path} done\n')

In [ ]:
for key, zip_path in zip_download_dict.items():
    simple_unzip(zip_path, dst_dir=zip_path.replace('.zip', ''))


In [ ]:
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

def process_dirs(root_dir):
    sub_dirs = os.listdir(root_dir)
    for idx, sub_name in enumerate(sub_dirs):
        sub_dir = os.path.join(root_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['input_1', 'input_2', 'input_3', 'input_4', 'input_5', 'input_6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, cam_name)
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                print(f'{image_dir_src} selecting...')
                image_dir_select = image_dir_src+'_select'
                shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
                select_img(image_dir_src, image_dir_select, gap=gap_num)
                print(f'{image_dir_select} filtering...')
                image_dir_filter = image_dir_src+'_filter'
                shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
                filter_deduplication(image_dir_select, image_dir_filter)
                print(f'{image_dir_filter} done\n')

In [ ]:
process_dirs(data_dir)

In [ ]:
def img_merge(input_dir, output_dir):
    sub_dirs = os.listdir(input_dir)
    if 'merge_dir' in sub_dirs:
        sub_dirs.remove('merge_dir')
    os.makedirs(output_dir, exist_ok=True)
    for sub_name in sub_dirs:
        sub_dir = os.path.join(input_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['input_1', 'input_2', 'input_3', 'input_4', 'input_5', 'input_6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, cam_name+'_filter')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                img_list = os.listdir(image_dir_src)
                for img_name in tqdm(img_list):
                    img_path_src = os.path.join(image_dir_src, img_name)
                    img_path_dst = os.path.join(output_dir, cam_name+'_'+img_name)
                    shutil.copyfile(img_path_src, img_path_dst)



In [ ]:
img_merge(data_dir, merge_dir)

In [ ]:
print(len(os.listdir(merge_dir)))

In [ ]:
import zipfile
import os

def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")



In [ ]:
def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")

In [ ]:
zip_folder_to_path(
    source_folder=os.path.join(merge_dir+'_infer', 'labels'),
    destination_zip=os.path.join(root_dir, os.path.basename(root_dir)+'_labels.zip')
)